In [32]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import (
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

import joblib
import os
import json
import warnings

warnings.filterwarnings("ignore")

In [33]:
# Load Data

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print(f"Shape: {df.shape}")

display(df.head())

Dataset loaded successfully
Shape: (1000, 48)


,Timestamp,Asset_ID,Latitude,Longitude,Inventory_Level,Shipment_Status,Temperature,Humidity,Traffic_Status,Waiting_Time,...,Utilization_Score,Waiting_Score,Customer_Value_Index,Customer_Value_Segment,Traffic_Level,Traffic_Utilization_Interaction,Demand_Traffic_Interaction,Utilization_Waiting_Interaction,Pre_Dispatch_Stress_Score,Distance_From_Center
0,2024-03-20 00:11:14,Truck_7,-65.7383,11.2497,390,Delayed,27.0,67.8,Detour,38,...,0.0025,0.56,1280,Medium Value,NaN,NaN,NaN,2283.8,0.46375,65.214852
1,2024-10-30 07:53:51,Truck_6,22.2748,-131.7086,491,In Transit,22.5,54.3,Heavy,16,...,0.5225,0.12,3073,Very High Value,NaN,NaN,NaN,1294.4,0.44625,134.636389
2,2024-07-29 18:42:48,Truck_10,54.9232,79.5455,190,In Transit,25.2,62.2,Detour,34,...,0.9800,0.48,1065,Medium Value,NaN,NaN,NaN,3372.8,0.89000,96.761714
3,2024-10-28 00:50:54,Truck_9,42.3900,-1.4788,330,Delivered,25.4,52.3,Heavy,37,...,0.9350,0.54,1135,Medium Value,NaN,NaN,NaN,3603.8,0.61750,43.811343
4,2024-09-27 15:52:58,Truck_7,-65.8477,47.9468,480,Delayed,20.5,57.2,Clear,56,...,0.2900,0.92,1182,Medium Value,NaN,NaN,NaN,4009.6,0.57000,79.862257


In [34]:
# Configuration

RANDOM_STATE = 42

TARGET = "Logistics_Delay"

TEST_SIZE = 0.20

RESULTS_DIR = "../results"
MODEL_DIR = "../models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Configuration loaded.")

Configuration loaded.


In [35]:
# Verify the existing train/test data

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTarget distribution - training:")
print(y_train.value_counts(normalize=True))

print("\nTarget distribution - test:")
print(y_test.value_counts(normalize=True))

X_train: (800, 44)
X_test : (200, 44)
y_train: (800,)
y_test : (200,)

Target distribution - training:
Logistics_Delay
1    0.5725
0    0.4275
Name: proportion, dtype: float64

Target distribution - test:
Logistics_Delay
1    0.54
0    0.46
Name: proportion, dtype: float64


In [36]:
# Identify numerical and categorical features

numeric_features = X_train.select_dtypes(
    include=["int64", "int32", "float64", "float32"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numeric_features)

Number of numerical features: 35
Number of categorical features: 9

Categorical features:
['Asset_ID', 'Traffic_Status', 'Month_Name', 'Utilization_Band', 'Inventory_Band', 'Purchase_Frequency_Band', 'Operational_Stress_Level', 'Time_Period', 'Customer_Value_Segment']

Numerical features:
['Latitude', 'Longitude', 'Inventory_Level', 'Temperature', 'Humidity', 'Waiting_Time', 'User_Transaction_Amount', 'User_Purchase_Frequency', 'Asset_Utilization', 'Demand_Forecast', 'Hour', 'Day', 'Day_of_Week', 'Month', 'Is_Weekend', 'Inventory_Coverage', 'Demand_Normalized', 'Utilization_Normalized', 'Waiting_Normalized', 'Operational_Stress_Score', 'Week_of_Year', 'Inventory_Demand_Gap', 'Stock_Risk', 'Fleet_Load_Index', 'Fleet_Load_Index_Normalized', 'Demand_Score', 'Utilization_Score', 'Waiting_Score', 'Customer_Value_Index', 'Traffic_Level', 'Traffic_Utilization_Interaction', 'Demand_Traffic_Interaction', 'Utilization_Waiting_Interaction', 'Pre_Dispatch_Stress_Score', 'Distance_From_Center']


In [37]:
# Build the optimization preprocessor

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

optimization_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

print("Optimization preprocessor created.")

Optimization preprocessor created.


In [38]:
# Verify preprocessing output

X_train_processed = optimization_preprocessor.fit_transform(X_train)
X_test_processed = optimization_preprocessor.transform(X_test)

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape :", X_test_processed.shape)

print(
    "NaN in processed X_train:",
    np.isnan(X_train_processed).sum()
)

print(
    "NaN in processed X_test:",
    np.isnan(X_test_processed).sum()
)

Processed X_train shape: (800, 77)
Processed X_test shape : (200, 77)
NaN in processed X_train: 0
NaN in processed X_test: 0


In [39]:
# Recreate the baseline Gradient Boosting pipeline

baseline_gb = GradientBoostingClassifier(
    random_state=RANDOM_STATE
)

baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", optimization_preprocessor),
        ("model", baseline_gb)
    ]
)

baseline_pipeline.fit(X_train, y_train)

baseline_pred = baseline_pipeline.predict(X_test)
baseline_prob = baseline_pipeline.predict_proba(X_test)[:, 1]

In [40]:
# Confirm baseline metrics

baseline_result = {
    "Model": "Gradient Boosting - Baseline",
    "Accuracy": accuracy_score(y_test, baseline_pred),
    "Precision": precision_score(y_test, baseline_pred, zero_division=0),
    "Recall": recall_score(y_test, baseline_pred, zero_division=0),
    "F1": f1_score(y_test, baseline_pred, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, baseline_prob),
    "PR_AUC": average_precision_score(y_test, baseline_prob)
}

baseline_result

{'Model': 'Gradient Boosting - Baseline',
 'Accuracy': 0.6,
 'Precision': 0.5972222222222222,
 'Recall': 0.7962962962962963,
 'F1': 0.6825396825396826,
 'ROC_AUC': 0.759963768115942,
 'PR_AUC': 0.853321039114266}

In [41]:
# Define Evaluation function

def evaluate_model(model_name, model, X_test, y_test):
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_test, y_pred
        ),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC_AUC": roc_auc_score(
            y_test, y_prob
        ),
        "PR_AUC": average_precision_score(
            y_test, y_prob
        )
    }

In [42]:
# Optimize Gradient Boosting

gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(optimization_preprocessor)),
        (
            "model",
            GradientBoostingClassifier(
                random_state=RANDOM_STATE
            )
        )
    ]
)

In [43]:
# Gradient Boosting parameter grid

gb_param_grid = {
    "model__n_estimators": [100, 150, 200],
    "model__learning_rate": [0.03, 0.05, 0.10],
    "model__max_depth": [2, 3],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__subsample": [0.8, 1.0]
}

print("Number of parameter combinations:")

from itertools import product

total_combinations = np.prod(
    [len(v) for v in gb_param_grid.values()]
)

print(total_combinations)

Number of parameter combinations:
144


In [44]:
from sklearn.model_selection import RandomizedSearchCV

In [45]:
gb_search = RandomizedSearchCV(
    estimator=gb_pipeline,
    param_distributions=gb_param_grid,
    n_iter=20,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

gb_search.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': [0.03, 0.05, ...], 'model__max_depth': [2, 3], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a functi

In [46]:
# Best Gradient Boosting parameters

print("Best Parameters:")
print(gb_search.best_params_)

print("\nBest CV F1:")
print(gb_search.best_score_)

Best Parameters:
{'model__subsample': 0.8, 'model__n_estimators': 200, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_depth': 2, 'model__learning_rate': 0.05}

Best CV F1:
0.7490083819787458


In [47]:
# Evaluate optimized Gradient Boosting

optimized_gb = gb_search.best_estimator_

gb_result = evaluate_model(
    "Gradient Boosting - Optimized",
    optimized_gb,
    X_test,
    y_test
)

gb_result

{'Model': 'Gradient Boosting - Optimized',
 'Accuracy': 0.695,
 'Precision': 0.7640449438202247,
 'Recall': 0.6296296296296297,
 'F1': 0.6903553299492385,
 'ROC_AUC': 0.7481884057971013,
 'PR_AUC': 0.8456405552923822}

In [48]:
# Compare baseline vs optimized Gradient Boosting

gb_comparison = pd.DataFrame(
    [
        baseline_result,
        gb_result
    ]
)

gb_comparison

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Gradient Boosting - Baseline,0.600,0.597222,0.796296,0.682540,0.759964,0.853321
1,Gradient Boosting - Optimized,0.695,0.764045,0.629630,0.690355,0.748188,0.845641


In [49]:
gb_comparison.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Gradient Boosting - Baseline,0.600,0.5972,0.7963,0.6825,0.7600,0.8533
1,Gradient Boosting - Optimized,0.695,0.7640,0.6296,0.6904,0.7482,0.8456


In [50]:
# Optimize Extra Trees

et_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(optimization_preprocessor)),
        (
            "model",
            ExtraTreesClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

In [51]:
# Extra Trees optimization

et_param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"]
}

et_search = RandomizedSearchCV(
    estimator=et_pipeline,
    param_distributions=et_param_grid,
    n_iter=15,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

et_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a

In [52]:
# Evaluate Extra Trees

optimized_et = et_search.best_estimator_

et_result = evaluate_model(
    "Extra Trees - Optimized",
    optimized_et,
    X_test,
    y_test
)

et_result

{'Model': 'Extra Trees - Optimized',
 'Accuracy': 0.72,
 'Precision': 0.8095238095238095,
 'Recall': 0.6296296296296297,
 'F1': 0.7083333333333334,
 'ROC_AUC': 0.7753119967793881,
 'PR_AUC': 0.8600555672851841}

In [53]:
# Optimize Random Forest

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(optimization_preprocessor)),
        (
            "model",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ]
)

In [54]:
# Random Forest Parameter search

rf_param_grid = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_grid,
    n_iter=15,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2], 'model__min_samples_split': [2, 5], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",15
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a

In [55]:
# Evaluate Random Forest

optimized_rf = rf_search.best_estimator_

rf_result = evaluate_model(
    "Random Forest - Optimized",
    optimized_rf,
    X_test,
    y_test
)

rf_result

{'Model': 'Random Forest - Optimized',
 'Accuracy': 0.725,
 'Precision': 0.7676767676767676,
 'Recall': 0.7037037037037037,
 'F1': 0.7342995169082126,
 'ROC_AUC': 0.7747584541062802,
 'PR_AUC': 0.8629923587441977}

In [56]:
# Optimize Logistic Regression

lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", clone(optimization_preprocessor)),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_STATE
            )
        )
    ]
)

In [57]:
# Logistic Regression optimization

lr_param_grid = {
    "model__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],
    "model__class_weight": [
        None,
        "balanced"
    ],
    "model__solver": [
        "lbfgs",
        "liblinear"
    ]
}

lr_search = RandomizedSearchCV(
    estimator=lr_pipeline,
    param_distributions=lr_param_grid,
    n_iter=10,
    scoring="f1",
    cv=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

lr_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__C': [0.01, 0.1, ...], 'model__class_weight': [None, 'balanced'], 'model__solver': ['lbfgs', 'liblinear']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``be

In [58]:
# Evaluate Logistic Regression

optimized_lr = lr_search.best_estimator_

lr_result = evaluate_model(
    "Logistic Regression - Optimized",
    optimized_lr,
    X_test,
    y_test
)

lr_result

{'Model': 'Logistic Regression - Optimized',
 'Accuracy': 0.74,
 'Precision': 0.9242424242424242,
 'Recall': 0.5648148148148148,
 'F1': 0.7011494252873564,
 'ROC_AUC': 0.7618760064412238,
 'PR_AUC': 0.8491623415867025}

In [59]:
# Collect all optimized models

optimized_models = {
    "Gradient Boosting": optimized_gb,
    "Extra Trees": optimized_et,
    "Random Forest": optimized_rf,
    "Logistic Regression": optimized_lr
}

optimized_models

{'Gradient Boosting': Pipeline(steps=[('preprocessor',
                  ColumnTransformer(transformers=[('num',
                                                   Pipeline(steps=[('imputer',
                                                                    SimpleImputer(strategy='median')),
                                                                   ('scaler',
                                                                    StandardScaler())]),
                                                   ['Latitude', 'Longitude',
                                                    'Inventory_Level',
                                                    'Temperature', 'Humidity',
                                                    'Waiting_Time',
                                                    'User_Transaction_Amount',
                                                    'User_Purchase_Frequency',
                                                    'Asset_Utilization',
            

In [60]:
# Optimized model comparison

optimized_results = []

for model_name, model in optimized_models.items():
    
    result = evaluate_model(
        model_name,
        model,
        X_test,
        y_test
    )
    
    optimized_results.append(result)

optimized_results_df = pd.DataFrame(
    optimized_results
)

optimized_results_df = optimized_results_df.sort_values(
    by="F1",
    ascending=False
).reset_index(drop=True)

optimized_results_df.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Random Forest,0.725,0.7677,0.7037,0.7343,0.7748,0.8630
1,Extra Trees,0.720,0.8095,0.6296,0.7083,0.7753,0.8601
2,Logistic Regression,0.740,0.9242,0.5648,0.7011,0.7619,0.8492
3,Gradient Boosting,0.695,0.7640,0.6296,0.6904,0.7482,0.8456


In [61]:
# Add baseline to comparison

comparison_df = pd.concat(
    [
        pd.DataFrame([baseline_result]),
        optimized_results_df
    ],
    ignore_index=True
)

comparison_df.round(4)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Gradient Boosting - Baseline,0.600,0.5972,0.7963,0.6825,0.7600,0.8533
1,Random Forest,0.725,0.7677,0.7037,0.7343,0.7748,0.8630
2,Extra Trees,0.720,0.8095,0.6296,0.7083,0.7753,0.8601
3,Logistic Regression,0.740,0.9242,0.5648,0.7011,0.7619,0.8492
4,Gradient Boosting,0.695,0.7640,0.6296,0.6904,0.7482,0.8456


In [62]:
# Rank Models - because this is a logistics prediction problem, F1 and Recall are particularly important.abs


comparison_df["F1_Rank"] = (
    comparison_df["F1"]
    .rank(
        ascending=False,
        method="min"
    )
)

comparison_df["PR_AUC_Rank"] = (
    comparison_df["PR_AUC"]
    .rank(
        ascending=False,
        method="min"
    )
)

comparison_df.sort_values(
    by=["F1", "PR_AUC"],
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,F1_Rank,PR_AUC_Rank
0,Random Forest,0.725,0.767677,0.703704,0.734300,0.774758,0.862992,1.0,1.0
1,Extra Trees,0.720,0.809524,0.629630,0.708333,0.775312,0.860056,2.0,2.0
2,Logistic Regression,0.740,0.924242,0.564815,0.701149,0.761876,0.849162,3.0,4.0
3,Gradient Boosting,0.695,0.764045,0.629630,0.690355,0.748188,0.845641,4.0,5.0
4,Gradient Boosting - Baseline,0.600,0.597222,0.796296,0.682540,0.759964,0.853321,5.0,3.0


In [63]:
# Select champion optimized model

champion_row = (
    optimized_results_df
    .sort_values(
        by=["F1", "PR_AUC"],
        ascending=False
    )
    .iloc[0]
)

champion_name = champion_row["Model"]

champion_model = optimized_models[
    champion_name
]

print("Champion Model:")
print(champion_name)

Champion Model:
Random Forest


In [82]:
# Threshold Optimization based on Recall

MIN_PRECISION = 0.70

eligible_thresholds = threshold_df[
    threshold_df["Precision"] >= MIN_PRECISION
].copy()

if len(eligible_thresholds) == 0:

    print("No threshold satisfies the minimum precision requirement.")

else:

    best_row = eligible_thresholds.loc[
        eligible_thresholds["Recall"].idxmax()
    ]

    recall_priority_threshold = best_row["Threshold"]

    print("=" * 60)
    print("RECALL-FIRST THRESHOLD OPTIMIZATION")
    print("=" * 60)

    print(f"Minimum Precision : {MIN_PRECISION:.2f}")
    print(f"Selected Threshold: {recall_priority_threshold:.2f}")
    print(f"Accuracy          : {best_row['Accuracy']:.4f}")
    print(f"Precision         : {best_row['Precision']:.4f}")
    print(f"Recall            : {best_row['Recall']:.4f}")
    print(f"F1                : {best_row['F1']:.4f}")

RECALL-FIRST THRESHOLD OPTIMIZATION
Minimum Precision : 0.70
Selected Threshold: 0.48
Accuracy          : 0.7050
Precision         : 0.7207
Recall            : 0.7407
F1                : 0.7306


In [ ]:
# Final predictions using recall-priority threshold

y_pred_recall = (
    y_prob_champion >= recall_priority_threshold
).astype(int)

In [84]:
# Final evaluation using optimized threshold

final_recall_result = {

    "Model": champion_name,

    "Threshold": recall_priority_threshold,

    "Accuracy": accuracy_score(
        y_test,
        y_pred_recall
    ),

    "Precision": precision_score(
        y_test,
        y_pred_recall,
        zero_division=0
    ),

    "Recall": recall_score(
        y_test,
        y_pred_recall,
        zero_division=0
    ),

    "F1": f1_score(
        y_test,
        y_pred_recall,
        zero_division=0
    ),

    "ROC_AUC": roc_auc_score(
        y_test,
        y_prob_champion
    ),

    "PR_AUC": average_precision_score(
        y_test,
        y_prob_champion
    )
}

final_recall_result

{'Model': 'Random Forest',
 'Threshold': np.float64(0.47999999999999976),
 'Accuracy': 0.705,
 'Precision': 0.7207207207207207,
 'Recall': 0.7407407407407407,
 'F1': 0.730593607305936,
 'ROC_AUC': 0.7747584541062802,
 'PR_AUC': 0.8629923587441977}

In [85]:
# Final confusion matrix

cm_recall = confusion_matrix(
    y_test,
    y_pred_recall
)

print("Confusion Matrix:")
print(cm_recall)

Confusion Matrix:
[[61 31]
 [28 80]]


In [86]:
print(
    classification_report(
        y_test,
        y_pred_recall,
        target_names=[
            "No Delay",
            "Delay"
        ],
        zero_division=0
    )
)

              precision    recall  f1-score   support

    No Delay       0.69      0.66      0.67        92
       Delay       0.72      0.74      0.73       108

    accuracy                           0.70       200
   macro avg       0.70      0.70      0.70       200
weighted avg       0.70      0.70      0.70       200



In [ ]:
# Compare F1-optimal vs Recall-priority

final_comparison = pd.DataFrame(
    [
        {
            "Stage": "Baseline Gradient Boosting",
            "Accuracy": baseline_result["Accuracy"],
            "Precision": baseline_result["Precision"],
            "Recall": baseline_result["Recall"],
            "F1": baseline_result["F1"],
            "ROC_AUC": baseline_result["ROC_AUC"],
            "PR_AUC": baseline_result["PR_AUC"]
        },
        {
            "Stage": "Optimized + Threshold Tuned",
            "Accuracy": final_result["Accuracy"],
            "Precision": final_result["Precision"],
            "Recall": final_result["Recall"],
            "F1": final_result["F1"],
            "ROC_AUC": final_result["ROC_AUC"],
            "PR_AUC": final_result["PR_AUC"]
        }
    ]
)

final_comparison.round(4)

,Stage,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Baseline Gradient Boosting,0.60,0.5972,0.7963,0.6825,0.7600,0.8533
1,Optimized + Threshold Tuned,0.77,0.9559,0.6019,0.7386,0.7748,0.8630


In [70]:
# Calculate improvement

improvement = {
    "Accuracy_Change":
        final_result["Accuracy"]
        - baseline_result["Accuracy"],

    "Precision_Change":
        final_result["Precision"]
        - baseline_result["Precision"],

    "Recall_Change":
        final_result["Recall"]
        - baseline_result["Recall"],

    "F1_Change":
        final_result["F1"]
        - baseline_result["F1"],

    "ROC_AUC_Change":
        final_result["ROC_AUC"]
        - baseline_result["ROC_AUC"],

    "PR_AUC_Change":
        final_result["PR_AUC"]
        - baseline_result["PR_AUC"]
}

pd.DataFrame(
    [improvement]
).round(4)

,Accuracy_Change,Precision_Change,Recall_Change,F1_Change,ROC_AUC_Change,PR_AUC_Change
0,0.17,0.3587,-0.1944,0.0561,0.0148,0.0097


In [71]:
# Feature importance

model_step = champion_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    
    feature_importance_values = (
        model_step.feature_importances_
    )
    
    feature_names = (
        champion_model
        .named_steps["preprocessor"]
        .get_feature_names_out()
    )
    
    feature_importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": feature_importance_values
    })
    
    feature_importance_df = (
        feature_importance_df
        .sort_values(
            by="Importance",
            ascending=False
        )
        .reset_index(drop=True)
    )
    
    feature_importance_df.head(20)

else:
    print(
        "Champion model does not provide "
        "feature_importances_."
    )

In [72]:
# Display top 20 features

feature_importance_df.head(20)

,Feature,Importance
0,cat__Traffic_Status_Heavy,0.152801
1,cat__Traffic_Status_Clear,0.050079
2,cat__Traffic_Status_Detour,0.042494
3,num__Latitude,0.029920
4,num__Inventory_Level,0.025460
5,num__Distance_From_Center,0.024654
6,num__User_Transaction_Amount,0.024473
7,num__Longitude,0.024092
8,num__Operational_Stress_Score,0.023842
9,num__Customer_Value_Index,0.023799


In [73]:
# Save final model

final_model_path = os.path.join(
    MODEL_DIR,
    "logistics_delay_champion_optimized.pkl"
)

joblib.dump(
    champion_model,
    final_model_path
)

print(
    f"Final model saved to:\n{final_model_path}"
)

Final model saved to:
../models\logistics_delay_champion_optimized.pkl


In [74]:
# Save threshold

threshold_path = os.path.join(
    MODEL_DIR,
    "logistics_delay_threshold.json"
)

with open(
    threshold_path,
    "w"
) as f:
    
    json.dump(
        {
            "threshold": float(best_threshold),
            "model": champion_name,
            "target": TARGET
        },
        f,
        indent=4
    )

print(
    f"Threshold saved to:\n{threshold_path}"
)

Threshold saved to:
../models\logistics_delay_threshold.json


In [75]:
# Save model comparison

comparison_path = os.path.join(
    RESULTS_DIR,
    "model_optimization_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print(
    f"Comparison saved to:\n{comparison_path}"
)

Comparison saved to:
../results\model_optimization_comparison.csv


In [76]:
# Save final evaluation

final_result_path = os.path.join(
    RESULTS_DIR,
    "final_model_evaluation.csv"
)

pd.DataFrame(
    [final_result]
).to_csv(
    final_result_path,
    index=False
)

print(
    f"Final evaluation saved to:\n{final_result_path}"
)

Final evaluation saved to:
../results\final_model_evaluation.csv


In [77]:
# Save feature  importance

if "feature_importance_df" in globals():
    
    feature_importance_path = os.path.join(
        RESULTS_DIR,
        "final_feature_importance.csv"
    )
    
    feature_importance_df.to_csv(
        feature_importance_path,
        index=False
    )
    
    print(
        f"Feature importance saved to:\n"
        f"{feature_importance_path}"
    )

Feature importance saved to:
../results\final_feature_importance.csv


In [78]:
# Final Summary

print("=" * 70)
print("FINAL MODEL OPTIMIZATION SUMMARY")
print("=" * 70)

print(f"\nBaseline Model:")
print("Gradient Boosting")

print("\nBaseline Performance:")
print(
    f"F1       : {baseline_result['F1']:.4f}"
)
print(
    f"ROC-AUC  : {baseline_result['ROC_AUC']:.4f}"
)
print(
    f"PR-AUC   : {baseline_result['PR_AUC']:.4f}"
)

print("\nChampion Optimized Model:")
print(champion_name)

print("\nOptimal Threshold:")
print(f"{best_threshold:.2f}")

print("\nFinal Performance:")
print(
    f"Accuracy : {final_result['Accuracy']:.4f}"
)
print(
    f"Precision: {final_result['Precision']:.4f}"
)
print(
    f"Recall   : {final_result['Recall']:.4f}"
)
print(
    f"F1       : {final_result['F1']:.4f}"
)
print(
    f"ROC-AUC  : {final_result['ROC_AUC']:.4f}"
)
print(
    f"PR-AUC   : {final_result['PR_AUC']:.4f}"
)

print("\n" + "=" * 70)

FINAL MODEL OPTIMIZATION SUMMARY

Baseline Model:
Gradient Boosting

Baseline Performance:
F1       : 0.6825
ROC-AUC  : 0.7600
PR-AUC   : 0.8533

Champion Optimized Model:
Random Forest

Optimal Threshold:
0.57

Final Performance:
Accuracy : 0.7700
Precision: 0.9559
Recall   : 0.6019
F1       : 0.7386
ROC-AUC  : 0.7748
PR-AUC   : 0.8630

